In [1]:
!pip install -q importlib-resources
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import *

# 1. Load SMALL ImageNet subset
train, test = tfds.load(
    "imagenette/160px-v2",
    split=["train[:500]","validation[:100]"],
    as_supervised=True
)

# 2. Preprocess
def prep(img, label):
    return preprocess_input(tf.image.resize(img,(224,224))), label

train = train.map(prep).batch(32)
test = test.map(prep).batch(32)

# 3. Pre-trained VGG16 feature extractor
base = VGG16(weights="imagenet", include_top=False)
extractor = Model(
    base.input,
    GlobalAveragePooling2D()(base.output)
)

# 4. Extract high-level features ONCE
Xtr = extractor.predict(train, verbose=0)
Xte = extractor.predict(test, verbose=0)

ytr = np.concatenate([y.numpy() for _,y in train])
yte = np.concatenate([y.numpy() for _,y in test])

print("Feature vector:", Xtr.shape)

# 5. Train new classification layer
clf = LogisticRegression(max_iter=200)
clf.fit(Xtr, ytr)

# 6. Test unseen images
pred = clf.predict(Xte)

# 7. Evaluation
print("\nAccuracy :", accuracy_score(yte,pred))
print("Precision:", precision_score(yte,pred,average="weighted"))
print("Recall   :", recall_score(yte,pred,average="weighted"))
print("F1-score :", f1_score(yte,pred,average="weighted"))

print("\nConfusion Matrix:")
print(confusion_matrix(yte,pred))

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imagenette/160px-v2/incomplete.A0J8FW_1.0.0/imagenette-train.tfrecord-[0-9…

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imagenette/160px-v2/incomplete.A0J8FW_1.0.0/imagenette-validation.tfrecord…

Dataset imagenette downloaded and prepared to /root/tensorflow_datasets/imagenette/160px-v2/1.0.0. Subsequent calls will reuse this data.
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Feature vector: (500, 512)

Accuracy : 0.98
Precision: 0.9815340909090909
Recall   : 0.98
F1-score : 0.9798028943326057

Confusion Matrix:
[[ 9  0  0  0  0  0  0  0  0  0]
 [ 0  9  0  0  0  0  0  0  0  0]
 [ 0  0 11  0  0  0  0  0  0  0]
 [ 0  0  0 15  0  0  0  0  0  0]
 [ 0  0  0  0  7  0  0  0  0  0]
 [ 0  0  0  1  0  9  0  0  0  0]
 [ 0  0  0  0  0  0 12  0  0  0]
 [ 0  0  0  0  0  0  0  7  0  1]
 [ 0  0  0  0  0  0  0  0  9  0]
 [ 0  0  0  0  0  0  0  0  0 10]]
